# Logistic regression — from scratch, then with PyTorch and sklearn

Companion notebook for **Eps 24–28** of the Logistic Regression series.

*Same training loop as linear regression. Two substitutions: a sigmoid in the forward pass, binary cross-entropy in place of MSE. By the end you'll have trained a binary classifier three ways.*

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 2. The dataset

We'll fake a spam dataset — two features per email (suspicious-word count, capitalization ratio), and a label (1 = spam, 0 = not).

Real spam detection uses thousands of features. Two is enough to *see* what's going on.

In [ ]:
n_per_class = 60

# Not-spam cluster (label 0): low on both features
not_spam = np.random.randn(n_per_class, 2) * 1.0 + np.array([1.5, 0.8])

# Spam cluster (label 1): high on both
spam = np.random.randn(n_per_class, 2) * 1.0 + np.array([5.0, 4.0])

X = np.vstack([not_spam, spam])
y = np.hstack([np.zeros(n_per_class), np.ones(n_per_class)])

print('X shape:', X.shape, ' y shape:', y.shape)
print('class balance:', int(y.sum()), 'spam ·', int((1 - y).sum()), 'not-spam')

### Plot it

In [ ]:
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=40, color='#00E5FF', label='not spam', edgecolor='black', linewidth=0.5)
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=40, color='#FF4081', label='spam',     edgecolor='black', linewidth=0.5)
plt.xlabel('suspicious-word count')
plt.ylabel('capitalization ratio')
plt.title('Spam vs not-spam (synthetic)')
plt.legend()
plt.show()

## 3. Try linear regression first — watch it break

The labels are 0 and 1. Linear regression fits any numbers — let's plug it in and see what we get.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_bad = LinearRegression()
lr_bad.fit(X, y)

# Predict for a few example points
examples = np.array([[0.5, 0.0], [3.0, 2.0], [6.0, 5.5], [10.0, 8.0]])
for ex in examples:
    pred = lr_bad.predict(ex.reshape(1, -1))[0]
    print(f'  features {ex} -> predicted label {pred:+.3f}')

# Some predictions are negative. Others are > 1. Probabilities are supposed to be in [0, 1].
# Linear regression doesn't know about that constraint.

## 4. The sigmoid function

Sigmoid squashes any real number into `(0, 1)`. It's the function we need.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Some values
for z in [-7, -2, 0, 2, 7]:
    print(f'  σ({z:+}) = {sigmoid(z):.4f}')

### Plot it

The S-shape. Asymptotes at 0 and 1 that it never reaches.

In [ ]:
zs = np.linspace(-10, 10, 200)
plt.plot(zs, sigmoid(zs), color='#FF4081', linewidth=2.5)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axhline(1, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.title('The sigmoid — squashes any real number into (0, 1)')
plt.show()

## 5. Log loss — the right loss for sigmoid outputs

MSE breaks here because sigmoid's flat tails kill the gradient when the model is confidently wrong. Log loss stays steep.

$$L = -\frac{1}{N}\sum_i \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

In [ ]:
def log_loss(y_true, y_pred):
    # Clip to avoid log(0)
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Test: if the truth is 1 and the model predicts...
for p in [0.99, 0.7, 0.5, 0.3, 0.01]:
    print(f'  truth=1, pred={p}  ->  log loss = {log_loss(np.array([1]), np.array([p])):.4f}')

### Plot the two loss curves

For true=1, the loss is `-log(p)`. Confidently right (p near 1) → tiny loss. Confidently wrong (p near 0) → loss blows up.

For true=0, mirrored — `-log(1-p)`.

In [ ]:
ps = np.linspace(0.001, 0.999, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ps, -np.log(ps), color='#FF4081', linewidth=2.5)
axes[0].set_xlabel('predicted probability p')
axes[0].set_ylabel('loss')
axes[0].set_title('truth = 1   →   loss = -log(p)')
axes[0].set_ylim(0, 6)

axes[1].plot(ps, -np.log(1 - ps), color='#00E5FF', linewidth=2.5)
axes[1].set_xlabel('predicted probability p')
axes[1].set_ylabel('loss')
axes[1].set_title('truth = 0   →   loss = -log(1 - p)')
axes[1].set_ylim(0, 6)

plt.tight_layout()
plt.show()

## 6. Train it by hand

Three weights this time: `w₁`, `w₂` (one per feature), and `b`. Gradient descent is the same loop as linear regression with the substitutions.

In [ ]:
def predict_prob(W, b, X):
    z = X @ W + b
    return sigmoid(z)

def grad_log(W, b, X, y):
    p = predict_prob(W, b, X)
    errs = p - y
    dW = (X.T @ errs) / len(y)
    db = np.mean(errs)
    return dW, db

# Initialize
W = np.zeros(2)
b = 0.0
lr = 0.1

history = []
for step in range(500):
    dW, db = grad_log(W, b, X, y)
    W -= lr * dW
    b -= lr * db
    history.append(log_loss(y, predict_prob(W, b, X)))

print(f'final W = {W.round(3)}   b = {b:.3f}   log loss = {history[-1]:.4f}')

### Loss curve

In [ ]:
plt.plot(history, color='#FF4081', linewidth=2)
plt.xlabel('step')
plt.ylabel('log loss')
plt.title('Training — loss going down over 500 iterations')
plt.show()

### The decision boundary

Plot the trained model's decision boundary over the data. For logistic regression, the boundary is the line where the predicted probability equals 0.5 — equivalently, where `W·x + b = 0`.

In [ ]:
# Boundary: w0·x0 + w1·x1 + b = 0  →  x1 = -(w0·x0 + b) / w1
x0_range = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
boundary_x1 = -(W[0] * x0_range + b) / W[1]

plt.scatter(X[y == 0, 0], X[y == 0, 1], s=40, color='#00E5FF', label='not spam', edgecolor='black', linewidth=0.5)
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=40, color='#FF4081', label='spam',     edgecolor='black', linewidth=0.5)
plt.plot(x0_range, boundary_x1, color='#FFD27F', linewidth=2.5, label='decision boundary')
plt.xlabel('suspicious-word count')
plt.ylabel('capitalization ratio')
plt.title('Logistic regression — decision boundary (probability = 0.5)')
plt.legend()
plt.show()

## 7. Same thing in PyTorch — the 5-line loop, two changes from linear regression

Same five-line training loop. Two substitutions: add a sigmoid in the forward pass, use binary cross-entropy as the loss.

`nn.BCEWithLogitsLoss` combines sigmoid + BCE in one call (more numerically stable than doing them separately).

In [ ]:
import torch
import torch.nn as nn

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).view(-1, 1)

model = nn.Linear(2, 1)   # 2 features in, 1 logit out
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()

# THE 5-LINE TRAINING LOOP
for step in range(500):
    logits = model(X_t)
    loss = loss_fn(logits, y_t)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

W_torch = model.weight.detach().numpy().flatten()
b_torch = model.bias.item()
print(f'PyTorch:  W={W_torch.round(3)}  b={b_torch:.3f}')
print(f'By-hand:  W={W.round(3)}  b={b:.3f}')

## 8. Same thing in sklearn — 3 lines

In [ ]:
from sklearn.linear_model import LogisticRegression

model_sk = LogisticRegression()
model_sk.fit(X, y)

print(f'sklearn coefficients:  {model_sk.coef_[0].round(3)}')
print(f'sklearn intercept:     {model_sk.intercept_[0]:.3f}')

# Predict probabilities for some example emails
examples = np.array([[0.5, 0.0], [3.0, 2.0], [6.0, 5.5]])
probs = model_sk.predict_proba(examples)
for ex, p in zip(examples, probs):
    print(f'  features {ex}  ->  P(not spam)={p[0]:.3f}  P(spam)={p[1]:.3f}')

## 9. What does the trained model say?

The coefficients tell you the *direction* and *strength* of each feature's effect on the log-odds of spam. Large positive coefficient → that feature pushes toward 'spam'.

In [ ]:
coef_word  = model_sk.coef_[0][0]
coef_caps  = model_sk.coef_[0][1]
intercept  = model_sk.intercept_[0]

print('The trained classifier says:')
print(f'  each extra suspicious word adds {coef_word:.2f} to the log-odds of spam')
print(f'  each unit of caps ratio adds   {coef_caps:.2f}')
print(f'  baseline log-odds (no features): {intercept:.2f}')

print()
print('In odds-multiplier terms (e^coef):')
print(f'  one more suspicious word multiplies the odds by  {np.exp(coef_word):.2f}x')
print(f'  one unit more caps ratio multiplies the odds by  {np.exp(coef_caps):.2f}x')

## 10. What we did

- Set up a binary classification problem (spam vs not-spam).
- Showed why linear regression fails — outputs go negative and above 1.
- Introduced the sigmoid (squash into `(0, 1)`) and log loss (the right loss for sigmoid outputs).
- Trained a logistic regression three ways: by-hand gradient descent, PyTorch's 5-line loop with sigmoid + BCE, and sklearn's 3 lines.
- All three agree.
- Read the trained model's coefficients as multiplicative effects on the odds.

**Next: classification metrics.** Confusion matrix, precision, recall, ROC. How to know whether your classifier is actually good.